SRS agent 로직 결과값 테스트.ipynb  
- SRS agent 로직 결과값 테스트.ipynb 부분 수정사항

	1. pass 1(초안 생성)에 포함되는 RAG 삭제 적용 필요 (프롬프트 입력값도 수정필요 - prompt / __init__.py)
	2. 생성물에 대한 검증 에이전트 적용 추가 해야함
	3. (node-chunk.py) 청크 갯수 설정 3 --> 20~30 변경 필요 ( 속도 너무 느림 )
	 

- 아직 기초 node나 함수 값들은 수정된게 아님 (모듈화된것에 필요한 것 적용 필요)
<쥬피터 python 파일 매칭 = 새로 생성, 삭제, 덮어쓰기 필요>
셀 A  경로 + 공통 유틸
      → 파일 없음 (주피터 전용 셋업)
      → extract_reqs는 어디에도 없는 헬퍼 (셀 전용)

셀 B  입력 로드 (RFP + 회의록)
      → test_run.py 의 load_rfp(), load_minutes()

셀 C  회의록 정제
      → utils/text_cleaner.py 의 clean_minutes()
      → nodes/normalize.py 가 이걸 호출

셀 D  RAG 검색 (블록별 + 오버랩)   ★ 삭제해서 저굥
      → nodes/rag.py
      → 지금 단순 cleaned_minutes[:500] 인데
        build_rag_context(블록별+오버랩)로 교체해야 함

셀 E  Pass1 초안 생성
      → nodes/pass1.py
      → source에 RFP id 부착 로직 반영

셀 F  중복 제거 (임베딩)           ★신규 추가
      → 기존 파일 없음, 새로 만들어야 함
      → nodes/dedup.py (신규) 또는 pass1.py 끝에 통합

셀 G  Pass2 검토                   ★ 삭제 필요 
      → nodes/pass2.py
      → 복구 로직 제거 + dedup_by_embedding 적용

셀 H  Safety grounding ★ 확인 필요 
      → nodes/safety.py
      → core/validator.py 의 is_grounded() 호출

셀 I  Merge ID 발급                
      → nodes/merge.py
      → 기존 RFP ID 유지 / 신규만 REQ 발급 로직

셀 J  docx 저장
      → services/docx_service.py 의 generate_docx()

### 경로 설정

In [1]:
import sys
import os
import logging

# 리눅스 환경에 맞게 경로 구분자를 '/'로 수정
ROOT = "/workspace"
sys.path.insert(0, f"{ROOT}/agents/req_agent/req_agent")
sys.path.insert(1, f"{ROOT}/data_pipeline")

# 작업 디렉토리 변경
os.chdir(f"{ROOT}/data_pipeline")

# 로그 설정
for name in ["httpx", "httpcore", "sentence_transformers", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

logging.basicConfig(level=logging.INFO, force=True) # force=True는 이미 설정된 로그가 있을 경우 초기화
logger = logging.getLogger(__name__)

# 공통 유틸 (요구사항 가져오기)
def extract_reqs(result):
    if not isinstance(result, dict):
        return []
    return result.get("requirements", result.get("requirement",[])) or []


print(f"✅ 경로/로그 설정 완료. 현재 작업 디렉토리: {os.getcwd()}")

✅ 경로/로그 설정 완료. 현재 작업 디렉토리: /workspace/data_pipeline


### 2. 입력 데이터 로드 (RFP+회의록)

In [2]:
import json
rfp_path = f"{ROOT}/data/requirement_sources/서민금융진흥원 AI기반 통합 플랫폼 구축 사업 제안요청서_전처리이후.json"
with open(rfp_path, encoding='utf-8') as f:
    rfp_data = json.load(f)

# print(rfp_data) # isinstance => dict 인지 확인 -> 확인되면 rfp_data['req']를 넣어라 아니면 원번 rfp_data
rfp = rfp_data['requirements'] if isinstance(rfp_data, dict) and "requirements" in rfp_data else rfp_data
# rfp


minutes_path = f"{ROOT}/data_pipeline/data/RFP_변경_회의록.txt"
with open(minutes_path, encoding='utf-8') as f:
    minutes = f.read()

print(f'RFP {len(rfp)}개 / 회의록 {len(minutes)} 자')
print(f"   첫 RFP: {rfp[0]['requirement_id']} — {rfp[0]['requirement_name']}")


RFP 62개 / 회의록 1657 자
   첫 RFP: COR-001 — 하도급


In [3]:
# !pip install pandas
# !pip install qdrant_client
# !pip install sentence_transformers

In [4]:
import pandas as pd

# 1. 각 요구사항의 description 글자 수를 추출하여 리스트로 저장
desc_lengths = []

for item in rfp:
    req_id = item.get("requirement_id", "UNKNOWN")
    # description 필드가 없거나 None일 경우를 대비해 예외 처리
    description_text = str(item.get("description", "")).strip()
    
    desc_lengths.append({
        "requirement_id": req_id,
        "desc_len": len(description_text)
    })

# 2. 판다스 데이터프레임으로 변환 후 통계 출력
df_desc = pd.DataFrame(desc_lengths)
stats_desc = df_desc["desc_len"].describe()

print("\n==================================================")
print("     🔍 서민금융진흥원 RFP [Description] 길이 분석     ")
print("==================================================")
print(f"📌 전체 요구사항 개수 : {len(df_desc)} 개")
print(f"🔹 평균 글자 수       : {stats_desc['mean']:.1f} 자")
print(f"🔹 가장 긴 글자 수    : {int(stats_desc['max'])} 자 (ID: {df_desc.loc[df_desc['desc_len'].idxmax(), 'requirement_id']})")
print(f"🔹 가장 짧은 글자 수  : {int(stats_desc['min'])} 자 (ID: {df_desc.loc[df_desc['desc_len'].idxmin(), 'requirement_id']})")
print(f"🔹 중앙값 (Median)    : {int(stats_desc['50%'])} 자")
print(f"🔹 상위 25% 지점 글자수: {int(stats_desc['75%'])} 자") 
print("==================================================")

# 3. 데이터 청킹 설정을 위한 가이드라인 출력
print("\n💡 [chunking.py 설정 튜닝 가이드]")
if stats_desc['mean'] > 1000:
    print("👉 Description의 평균 길이가 긴 편입니다.")
    print("   2개씩 묶어서 처리(MAX_ITEMS_PER_CHUNK=2)하려면,")
    print("   MAX_CHARS_PER_CHUNK 설정을 최소 3500 ~ 4000 자 이상으로 늘려주는 것이 안전합니다.")
else:
    print("👉 Description의 평균 길이가 적당합니다.")
    print("   현재 설정된 MAX_CHARS_PER_CHUNK = 2500 제한 속에서 요구사항들이 찢어지지 않고")
    print("   2개씩 예쁘게 묶여서 처리될 가능성이 높습니다.")
print("==================================================")


     🔍 서민금융진흥원 RFP [Description] 길이 분석     
📌 전체 요구사항 개수 : 62 개
🔹 평균 글자 수       : 481.1 자
🔹 가장 긴 글자 수    : 1545 자 (ID: COR-001)
🔹 가장 짧은 글자 수  : 64 자 (ID: PMR-003)
🔹 중앙값 (Median)    : 343 자
🔹 상위 25% 지점 글자수: 702 자

💡 [chunking.py 설정 튜닝 가이드]
👉 Description의 평균 길이가 적당합니다.
   현재 설정된 MAX_CHARS_PER_CHUNK = 2500 제한 속에서 요구사항들이 찢어지지 않고
   2개씩 예쁘게 묶여서 처리될 가능성이 높습니다.


### step1 : 회의록 정제(normalize)

In [5]:
from utils.text_cleaner import clean_meeting_minutes

cleaned_minutes = clean_meeting_minutes(minutes)
print(f"정제 전: {len(minutes)}자")
print(f"정제 후: {len(cleaned_minutes)}자")
print(f"앞부분: {cleaned_minutes[:300]}")



정제 전: 1657자
정제 후: 1420자
앞부분: 회의록

회의일: 2026년 7월 15일 오전 10시 
참석자: 최발주(발주처), 이기획(PM), 김개발(개발팀장)

1. 발주처 변경 요청 사항
1) AI 모델 성능 향상 요구 
- 현재 AI 분석 모델의 신용평가 정확도를 기존 85%에서 92% 이상으로 상향 조정 요청 
- 금융 취약계층의 신용 리스크를 보다 정확히 판단하여 맞춤형 지원 강화 목적
2) 사용자 인터페이스(UI) 간소화 및 다국어 지원 추가 
- 서민 사용자 편의를 위해 UI를 더욱 직관적이고 간단하게 개선 요청 
- 다국어(영어, 중국어) 지원 추가로 외국인 서


### 공통 유틸

### Step2: 회의록과 유사한 context 추출
- RAG 검색 (유사 문서)

In [6]:
# !pip uninstall -y transformers torchvision
# !pip install torchvision --upgrade
# !pip install transformers==4.44.0  # 5.x에서 문제가 지속된다면 안정화된 4.44 버전을 권장합니다.
# !pip install sentence-transformers

In [7]:
from services.rag_service import RAGService
import re
# 오버랩 추가
def split_with_overlap(text, max_chars = 800, overlap=200):
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    step = max_chars - overlap
    while start < len(text):
        chunk = text[start:start+ max_chars].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

def build_rag_context(rag, minutes, max_chars = 1000, overlap = 200, max_results = 12):
    blocks = [b.strip() for b in re.split(r"\n\s*\n", minutes) if b.strip()] or [minutes]
    queries = []
    for b in blocks:
        queries.extend(split_with_overlap(b, max_chars, overlap))

    merged, seen = [], set()
    for q in queries:
        for hit in (rag.query(q) or []):
            sig = (hit.get("source",""), hit.get("text",""))
            if sig in seen:
                continue
            seen.add(sig)
            merged.append(hit)

    merged.sort(key=lambda h: h.get("score",0), reverse=True)
    merged = merged[:max_results]
    return merged, rag.format_context(merged)

rag = RAGService()
rag_results, rag_context = build_rag_context(rag, cleaned_minutes, max_chars= 800, overlap=200)
print(f"✅ RAG {len(rag_results)}건 / 컨텍스트 {len(rag_context)}자")
# print(rag_context)


📥 로컬 임베딩 모델 로딩 중...


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


✅ 임베딩 모델 로드 완료.
✅ RAG 12건 / 컨텍스트 6935자


### PASS1 초안 생성 (LLM, RFP 청크)

1. [회의록 over lap 적용 추출]  ➔ [회의록 유사 내용 Qdrant 검색] ➔ [RAG 데이터] 획득
2. [RFP] + [회의록] + [RAG 데이터] ➔ build_pass1_prompt() 실행 ➔ LLM 호출 ➔ [초안] 생성
3. [RFP] + [회의록] + [RAG 데이터] + [초안] ➔ build_pass2_prompt() 실행 ➔ LLM 호출 ➔ [최종 결과물] 완성

In [8]:
from services.llm_service import LLMService
from prompts import GENERATION_SYSTEM, build_pass1_prompt
from nodes.chunking import chunk_items, compact_text

llm = LLMService()

def pass1_generate(llm, rfp_items, munutes, rag_context):
    draft = []
    for i, chunk in enumerate(chunk_items(rfp_items)):
        ids = [r.get("requirement_id", "") for r in chunk if r.get("requirement_id")]
        print(f"Pass1 청크 {i} : {ids}")
        result = llm.complete_json(
            GENERATION_SYSTEM,
            build_pass1_prompt(chunk, compact_text(minutes), compact_text(rag_context)),
        )
        reqs = extract_reqs(result)
        for r in reqs:
            src = r.get("source",[]) or []
            for rid in ids:
                if rid and rid not in src:
                    src.append(rid)
            r["source"] = src
        draft.extend(reqs)
    return draft

test_rfp = rfp # 테스트 3개 
# test_rfp = rfp[:3] # 테스트 3개 
draft_reqs = pass1_generate(llm, test_rfp, cleaned_minutes, rag_context)
print(f"\n✅ 초안 {len(draft_reqs)}개")

Pass1 청크 0 : ['COR-001', 'ECR-001', 'ECR-002']
Pass1 청크 1 : ['ECR-003', 'SFR-001', 'SFR-002']
Pass1 청크 2 : ['SFR-003', 'SFR-004', 'SFR-005']
Pass1 청크 3 : ['SFR-006', 'SFR-007', 'SFR-008']
Pass1 청크 4 : ['SFR-009', 'SFR-010', 'SFR-011']
Pass1 청크 5 : ['SFR-012', 'SFR-013', 'DAR-001']
Pass1 청크 6 : ['DAR-002', 'DAR-003', 'DAR-004']
Pass1 청크 7 : ['SER-001', 'SER-002', 'SER-003']
Pass1 청크 8 : ['SER-004', 'SER-005', 'SER-006']


ERROR:core.parser:parser: all strategies failed. raw={
  "requirements": [
    {
      "requirement_id": "SER-004",
      "requirement_name": "시스템 구축 보안요건",
      "requirement_type": "보안",
      "description": "시스템 구축 보안요건 \nㅇ 동 구축과 관련 서금원 보안 아키텍처를 고려하여
  "requirements": [
    {
      "requirement_id": "SER-004",
      "requirement_name": "시스템 구축 보안요건",
      "requirement_type": "보안",
      "description": "시스템 구축 보안요건 \nㅇ 동 구축과 관련 서금원 보안 아키텍처를 고려하여


Pass1 청크 9 : ['COR-002', 'COR-003', 'COR-004']
Pass1 청크 10 : ['COR-005', 'COR-006', 'PER-001']
Pass1 청크 11 : ['PER-002', 'PER-003', 'PER-004']
Pass1 청크 12 : ['SIR-001', 'SIR-002', 'SIR-003']
Pass1 청크 13 : ['SIR-004', 'QUR-001', 'QUR-002']
Pass1 청크 14 : ['QUR-003', 'QUR-004', 'QUR-005']
Pass1 청크 15 : ['QUR-006', 'TER-001', 'TER-002']
Pass1 청크 16 : ['TER-003', 'TER-004', 'PMR-001']


ERROR:core.parser:parser: all strategies failed. raw={
  "requirements": [
    {
      "requirement_id": "TER-003",
      "requirement_name": "인수 테스트 수행",
      "requirement_type": "테스트",
      "description": "인수 시험에 관한 사항\n◦ 시스템 설치 및 테스트 완료 후에 계약상대자는 인
  "requirements": [
    {
      "requirement_id": "TER-003",
      "requirement_name": "인수 테스트 수행",
      "requirement_type": "테스트",
      "description": "인수 시험에 관한 사항\n◦ 시스템 설치 및 테스트 완료 후에 계약상대자는 인


Pass1 청크 17 : ['PMR-002', 'PMR-003', 'PMR-004']
Pass1 청크 18 : ['PMR-005', 'PMR-006', 'PSR-001']
Pass1 청크 19 : ['PSR-002', 'PSR-003', 'PSR-004']
Pass1 청크 20 : ['PSR-005', 'PSR-006']

✅ 초안 106개


#### 내용 기반 중복 제거

In [9]:
from phase1_config import get_embedding
import numpy as np

def dedup_by_embedding(reqs, threshold=0.90):
    kept, vecs = [], []
    for r in reqs:
        desc = r.get("description", "")
        if not desc:
            kept.append(r); vecs.append(None); continue
        vec = np.array(get_embedding(desc))
        is_dup = False
        for i, kv in enumerate(vecs):
            if kv is None: continue
            if float(np.dot(vec, kv)) >= threshold:
                kept[i]["source"] = list(dict.fromkeys((kept[i].get("source") or []) + (r.get("source") or [])))
                is_dup = True
                break
        if not is_dup:
            kept.append(r); vecs.append(vec)
    return kept


draft_reqs = dedup_by_embedding(draft_reqs, threshold=0.90)
print(f"✅ 중복 제거 후 {len(draft_reqs)}개")
    

✅ 중복 제거 후 69개


In [10]:
draft_reqs

[{'requirement_id': 'COR-001',
  'requirement_name': '하도급',
  'requirement_type': '제약사항',
  'description': '공통 요구사항 \nㅇ 본 사업에서 제시한 사업요건을 준수하여 제안하고, 사업 내용이 변경될 경우 상호 협의 하에 결정 ㅇ 제안요청서 요구사항에 명시되지 않았으나 제안업체의 판단으로 추가가 필요한 요구사항에 대하여 제안서에 별도 표시하여 추가 작성 가능 ㅇ 제안업체는 제안 요청사항을 상세히 검토하여 상기 제안요청 사항의 미비점이나 부족한 사항을 보완하고, 전체 시스템의 완전한 가용성, 안정성, 운영 용이성을 고려하여 제안서를 작성하여야 함 ㅇ 제안요청서의 각 부문에서 제시된 내용의 항목별 수용여부를 제시하여야 하며, 수용하지 못하거나 일부만 수용 가능한 항목에 대하여 사유를 상세히 표기하여야 함 ㅇ 시스템 구축을 위해 필요한 환경설정, 구성변경 등 모든 제반사항은 제안사의 책임으로 수행하되 반드시 진흥원과 협의하여야 함 ㅇ 시스템 이관에 필요한 사항 일체에 대하여 제안업체의 책임으로 수행하여야 함 ㅇ 시스템 설치·연동·이관에 필요한 제조사 및 유지보수 업체의 지원은 제안사가 수행 ㅇ 사업 수행과정에서 발생되는 제반 비용은 제안업체가 부담하여야 하며, 사업 목적 달성을 위해 별도 명시하지 않은 사항이라도 필요한 모든 구성을 제안하여야 함 - 이를 간과하여 사업 수행에 차질이 발생한 경우 별도비용 청구 없이 제안업체의 부담으로 보완 ㅇ 업무 시스템(채널계 포함)과 연동되는 AI 기능은 AI 서비스 장애 또는 중단 시에도 기존 업무 시스템의 정상적인 이용에 영향을 주지 않아야 함 ㅇ AI 기능은 업무 시스템과 비동기 또는 분리 구조로 연계되어야 하며, AI 서비스 미가용 시에도 기존 시스템 기능은 기본 프로세스로 대체 수행될 수 있어야 함',
  'source': ['서민금융진흥원 AI기반 통합 플랫폼 구축 사업 제안요청서.docx',
   'COR-001',
   'ECR-0

### STEP 4 : pass 2 검토/보완 

1. 프롬프트(REFINE_SYSTEM)에 제약 조건 딱 한 줄 추가 (가장 중요)
LLM이 정제 작업을 할 때 원본 ID를 건드리지 못하게 머릿속에 박아줘야 합니다. prompts.py 파일의 REFINE_SYSTEM 내용 중 제약조건(Constraints) 영역에 아래 문구를 추가해 주세요.

추가할 프롬프트 문구

"- 입력받은 초안 중 기존 RFP 고유 ID(예: COR-001, ECR-001 등)를 이미 가지고 있는 항목은 절대로 삭제하거나 누락하지 말고 반드시 결과에 포함시키시오. 회의록 내용이 반영되어 쪼개지더라도 원본 ID 항목은 그대로 유지해야 합니다."

In [11]:
from prompts import REFINE_SYSTEM, build_pass2_prompt

def refine(llm, rfp_items, minutes, rag_context, draft_reqs, chunk_size=3):
    refined = []

    for i in range(0, len(draft_reqs), chunk_size):
        chunk = draft_reqs[i:i+chunk_size]
        result = llm.complete_json(
            REFINE_SYSTEM,
            build_pass2_prompt(rfp_items, minutes, rag_context, chunk),)
        reqs = extract_reqs(result)
        if not reqs:
            logger.warning(f"청크 {i} 정제 없음 → 원본 유지")
            refined.extend(chunk)
            continue
        refined.extend(reqs)
    return dedup_by_embedding(refined, threshold=0.90)

refined_reqs = refine(llm, test_rfp, cleaned_minutes, rag_context, draft_reqs)
print(f"✅ 검토 완료 {len(refined_reqs)}개")

ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all strategies failed. raw=[


ERROR:core.parser:parser: all s

✅ 검토 완료 69개


In [12]:
print("\n" + "="*40 + " 🔄 요구사항 정제(Pass 2) 비교 리포트 " + "="*40)

# 비교를 편하게 하기 위해 정제된 요구사항들을 이름(name) 기준으로 딕셔너리화
refined_dict = {req.get("requirement_name"): req for req in refined_reqs if req.get("requirement_name")}

for idx, draft in enumerate(draft_reqs, 1):
    name = draft.get("requirement_name")
    draft_desc = draft.get("description", "").strip()
    
    print(f"\n[{idx}] 요구사항명: {name} (ID: {draft.get('requirement_id') or '신규'})")
    print(f"  ❌ 변경 전 (Pass 1 초안):\n    " + draft_desc.replace('\n', '\n    '))
    
    # Pass 2 결과물에 같은 이름의 요구사항이 있는지 확인
    if name in refined_dict:
        refined_desc = refined_dict[name].get("description", "").strip()
        
        if draft_desc == refined_desc:
            print("  🟢 변경 없음 (원본 문장 유지됨)")
        else:
            print(f"  ✨ 변경 후 (Pass 2 정제):\n    " + refined_desc.replace('\n', '\n    '))
    else:
        # 이름이 사라졌다면 병합되었거나 삭제된 것
        print("  🚨 상태 변경: Pass 2 단계에서 다른 요구사항과 병합되었거나 삭제됨")

print("\n" + "="*112)


======================================== 🔄 요구사항 정제(Pass 2) 비교 리포트 ========================================

[1] 요구사항명: 하도급 (ID: COR-001)
  ❌ 변경 전 (Pass 1 초안):
    공통 요구사항 
    ㅇ 본 사업에서 제시한 사업요건을 준수하여 제안하고, 사업 내용이 변경될 경우 상호 협의 하에 결정 ㅇ 제안요청서 요구사항에 명시되지 않았으나 제안업체의 판단으로 추가가 필요한 요구사항에 대하여 제안서에 별도 표시하여 추가 작성 가능 ㅇ 제안업체는 제안 요청사항을 상세히 검토하여 상기 제안요청 사항의 미비점이나 부족한 사항을 보완하고, 전체 시스템의 완전한 가용성, 안정성, 운영 용이성을 고려하여 제안서를 작성하여야 함 ㅇ 제안요청서의 각 부문에서 제시된 내용의 항목별 수용여부를 제시하여야 하며, 수용하지 못하거나 일부만 수용 가능한 항목에 대하여 사유를 상세히 표기하여야 함 ㅇ 시스템 구축을 위해 필요한 환경설정, 구성변경 등 모든 제반사항은 제안사의 책임으로 수행하되 반드시 진흥원과 협의하여야 함 ㅇ 시스템 이관에 필요한 사항 일체에 대하여 제안업체의 책임으로 수행하여야 함 ㅇ 시스템 설치·연동·이관에 필요한 제조사 및 유지보수 업체의 지원은 제안사가 수행 ㅇ 사업 수행과정에서 발생되는 제반 비용은 제안업체가 부담하여야 하며, 사업 목적 달성을 위해 별도 명시하지 않은 사항이라도 필요한 모든 구성을 제안하여야 함 - 이를 간과하여 사업 수행에 차질이 발생한 경우 별도비용 청구 없이 제안업체의 부담으로 보완 ㅇ 업무 시스템(채널계 포함)과 연동되는 AI 기능은 AI 서비스 장애 또는 중단 시에도 기존 업무 시스템의 정상적인 이용에 영향을 주지 않아야 함 ㅇ AI 기능은 업무 시스템과 비동기 또는 분리 구조로 연계되어야 하며, AI 서비스 미가용 시에도 기존 시스템 기능은 기본 프로세스로 대체 수행될 수 있어야 함
  🟢 변경 없음 (원본 문장 유지

In [13]:
refined_reqs

[{'requirement_id': 'COR-001',
  'requirement_name': '하도급',
  'requirement_type': '제약사항',
  'description': '공통 요구사항 \nㅇ 본 사업에서 제시한 사업요건을 준수하여 제안하고, 사업 내용이 변경될 경우 상호 협의 하에 결정 ㅇ 제안요청서 요구사항에 명시되지 않았으나 제안업체의 판단으로 추가가 필요한 요구사항에 대하여 제안서에 별도 표시하여 추가 작성 가능 ㅇ 제안업체는 제안 요청사항을 상세히 검토하여 상기 제안요청 사항의 미비점이나 부족한 사항을 보완하고, 전체 시스템의 완전한 가용성, 안정성, 운영 용이성을 고려하여 제안서를 작성하여야 함 ㅇ 제안요청서의 각 부문에서 제시된 내용의 항목별 수용여부를 제시하여야 하며, 수용하지 못하거나 일부만 수용 가능한 항목에 대하여 사유를 상세히 표기하여야 함 ㅇ 시스템 구축을 위해 필요한 환경설정, 구성변경 등 모든 제반사항은 제안사의 책임으로 수행하되 반드시 진흥원과 협의하여야 함 ㅇ 시스템 이관에 필요한 사항 일체에 대하여 제안업체의 책임으로 수행하여야 함 ㅇ 시스템 설치·연동·이관에 필요한 제조사 및 유지보수 업체의 지원은 제안사가 수행 ㅇ 사업 수행과정에서 발생되는 제반 비용은 제안업체가 부담하여야 하며, 사업 목적 달성을 위해 별도 명시하지 않은 사항이라도 필요한 모든 구성을 제안하여야 함 - 이를 간과하여 사업 수행에 차질이 발생한 경우 별도비용 청구 없이 제안업체의 부담으로 보완 ㅇ 업무 시스템(채널계 포함)과 연동되는 AI 기능은 AI 서비스 장애 또는 중단 시에도 기존 업무 시스템의 정상적인 이용에 영향을 주지 않아야 함 ㅇ AI 기능은 업무 시스템과 비동기 또는 분리 구조로 연계되어야 하며, AI 서비스 미가용 시에도 기존 시스템 기능은 기본 프로세스로 대체 수행될 수 있어야 함',
  'source': ['서민금융진흥원 AI기반 통합 플랫폼 구축 사업 제안요청서.docx',
   'COR-001',
   'ECR-0

Pass 1 (초안 생성): 회의록과 RFP를 다 긁어모으는 "수집 단계"입니다. 여기서 근거 검증을 하면, 아직 문장이 덜 다듬어져 있어서 LLM이 "근거가 없다"고 오판할 가능성이 높습니다.

Pass 2 (정제): 수집된 원석들을 ~하여야 한다는 표준 문체로 다듬고, 중복을 제거하고, ID를 관리하는 "가공 단계"입니다. 이 단계에서 데이터의 모양이 비로소 완성됩니다.

Pass 3 (근거 검증 - is_grounded): 이제 완성된 "최종 정제본"을 가지고, "이 문장이 진짜 회의록이나 RFP에 근거를 두고 있는가?"를 팩트 체크하는 "검수 단계"입니다.

In [14]:
from core.validator import is_grounded

validated_reqs = []
for req in refined_reqs:
    r = is_grounded(req, cleaned_minutes, rag)
    req["_grounded"] = r.is_grounded
    req["_score"]    = r.score
    req["_reason"]   = r.reason
    validated_reqs.append(req)
    flag = "✅" if r.is_grounded else "⚠️"
    print(f"{flag} {req.get('requirement_name','?')[:20]:<20} score={r.score:.2f}")

validated_reqs

✅ 하도급                  score=0.44
✅ 인프라 구축(공통)           score=0.40
✅ S/W 구축(공통)           score=0.42
✅ AI 모델 성능 향상          score=0.88
✅ UI 간소화 및 다국어 지원      score=1.00
✅ 보안 정책 강화 및 실시간 이상거래  score=0.95
✅ 생성형 AI 주요 장비         score=0.38
✅ 생성형 AI 기본사항          score=0.35
✅ AI 적용 모델             score=0.37
✅ AI 모델 성능 향상          score=0.87
✅ UI 간소화 및 다국어 지원      score=0.67
✅ 보안 정책 강화 및 실시간 이상거래  score=0.77
✅ LLM 개발 및 운영 관리 기본사항  score=0.39
✅ 검색증강생성(RAG) 기본사항 (RA score=0.35
✅ AI Agent 개발 및 운영 관리  score=0.34
✅ AI 응용 서비스 운영 관리 기본사항 score=0.37
✅ ML 모델 학습·실험 운영 관리 (M score=0.37
✅ AI OCR 기술기반 자동화      score=0.34
✅ 생성형 AI 활용을 위한 API 제공 score=0.37
✅ AI 서비스 주요 기능         score=0.39
✅ 생성형 AI 포털 구축         score=0.34
✅ 보안 정책 강화 및 실시간 이상거래  score=0.92
✅ 데이터 마트 고도화           score=0.36
✅ AI 컨설팅               score=0.35
✅ 문서 기반 학습 데이터 수집 및 관리 score=0.38
✅ 문서 승인 기반 RAG 학습 자동화  score=0.37
✅ 데이터 정제 및 관리          score=0.43
✅ 데이터 품질관리 및 표준 준수     score=0.40
✅ 보안 정책 강화 및 실시간 이상거래  score=0.91
✅ 보안관리 및 규정 준수

[{'requirement_id': 'COR-001',
  'requirement_name': '하도급',
  'requirement_type': '제약사항',
  'description': '공통 요구사항 \nㅇ 본 사업에서 제시한 사업요건을 준수하여 제안하고, 사업 내용이 변경될 경우 상호 협의 하에 결정 ㅇ 제안요청서 요구사항에 명시되지 않았으나 제안업체의 판단으로 추가가 필요한 요구사항에 대하여 제안서에 별도 표시하여 추가 작성 가능 ㅇ 제안업체는 제안 요청사항을 상세히 검토하여 상기 제안요청 사항의 미비점이나 부족한 사항을 보완하고, 전체 시스템의 완전한 가용성, 안정성, 운영 용이성을 고려하여 제안서를 작성하여야 함 ㅇ 제안요청서의 각 부문에서 제시된 내용의 항목별 수용여부를 제시하여야 하며, 수용하지 못하거나 일부만 수용 가능한 항목에 대하여 사유를 상세히 표기하여야 함 ㅇ 시스템 구축을 위해 필요한 환경설정, 구성변경 등 모든 제반사항은 제안사의 책임으로 수행하되 반드시 진흥원과 협의하여야 함 ㅇ 시스템 이관에 필요한 사항 일체에 대하여 제안업체의 책임으로 수행하여야 함 ㅇ 시스템 설치·연동·이관에 필요한 제조사 및 유지보수 업체의 지원은 제안사가 수행 ㅇ 사업 수행과정에서 발생되는 제반 비용은 제안업체가 부담하여야 하며, 사업 목적 달성을 위해 별도 명시하지 않은 사항이라도 필요한 모든 구성을 제안하여야 함 - 이를 간과하여 사업 수행에 차질이 발생한 경우 별도비용 청구 없이 제안업체의 부담으로 보완 ㅇ 업무 시스템(채널계 포함)과 연동되는 AI 기능은 AI 서비스 장애 또는 중단 시에도 기존 업무 시스템의 정상적인 이용에 영향을 주지 않아야 함 ㅇ AI 기능은 업무 시스템과 비동기 또는 분리 구조로 연계되어야 하며, AI 서비스 미가용 시에도 기존 시스템 기능은 기본 프로세스로 대체 수행될 수 있어야 함',
  'source': ['서민금융진흥원 AI기반 통합 플랫폼 구축 사업 제안요청서.docx',
   'COR-001',
   'ECR-0

In [15]:
from core.id_manager import assign_ids

_OUTPUT_KEYS = ["requirement_id", "requirement_name", "requirement_type",
                "description", "source", "constraints", "priority",
                "validation_criteria", "note", "status"]

# 1. 기존/신규 데이터를 명확히 분리
existing_reqs = []
new_reqs = []

for r in validated_reqs:
    rid = (r.get("requirement_id") or "").strip()
    r.setdefault("source", ["generated"])
    
    if rid:
        r["status"] = '기존'
        existing_reqs.append(r)
    else:
        r["requirement_id"] = ""  # ID 할당기에서 새로 부여할 수 있도록 비워둠
        r["status"] = "신규"
        new_reqs.append(r)

# 2. 신규 항목에만 ID 할당 (기존 데이터는 보호)
assigned_new = assign_ids([], new_reqs, prefix="REQ")

# 3. 다시 합치기
all_reqs = existing_reqs + assigned_new

# 4. 최종 리스트 생성 (필드 필터링)
final_reqs = [{k: r.get(k) for k in _OUTPUT_KEYS} for r in all_reqs]

print(f"✅ 최종 {len(final_reqs)}개")
for r in final_reqs:
    print(f"   {r['requirement_id']:<10} | {r['status']:<4} | {r['requirement_name'][:20]}")

✅ 최종 69개
   COR-001    | 기존   | 하도급
   ECR-001    | 기존   | 인프라 구축(공통)
   ECR-002    | 기존   | S/W 구축(공통)
   ECR-003    | 기존   | 생성형 AI 주요 장비
   SFR-001    | 기존   | 생성형 AI 기본사항
   SFR-002    | 기존   | AI 적용 모델
   SFR-003    | 기존   | LLM 개발 및 운영 관리 기본사항 
   SFR-004    | 기존   | 검색증강생성(RAG) 기본사항 (RA
   SFR-005    | 기존   | AI Agent 개발 및 운영 관리 
   SFR-006    | 기존   | AI 응용 서비스 운영 관리 기본사항
   SFR-007    | 기존   | ML 모델 학습·실험 운영 관리 (M
   SFR-008    | 기존   | AI OCR 기술기반 자동화
   SFR-009    | 기존   | 생성형 AI 활용을 위한 API 제공
   SFR-010    | 기존   | AI 서비스 주요 기능
   SFR-011    | 기존   | 생성형 AI 포털 구축
   SFR-012    | 기존   | 데이터 마트 고도화
   SFR-013    | 기존   | AI 컨설팅
   DAR-001    | 기존   | 문서 기반 학습 데이터 수집 및 관리
   DAR-002    | 기존   | 문서 승인 기반 RAG 학습 자동화
   DAR-003    | 기존   | 데이터 정제 및 관리
   DAR-004    | 기존   | 데이터 품질관리 및 표준 준수
   SER-001    | 기존   | 보안관리 및 규정 준수
   SER-002    | 기존   | 개인정보보호 준수사항
   SER-003    | 기존   | 보안성 검토 및 보안취약점 점검·조치
   COR-002    | 기존   | 일반사항
   COR-003    | 기존   | 개발 표준 준수
   COR-004    | 기

In [16]:
# 최종 실행 코드
from services.docx_service import generate_docx
# 1. 정제된 final_reqs 데이터를 넘겨서 1개짜리 통합 표 기반의 단일 문서 생성
path = generate_docx(final_reqs, prefix="사용자 요구사항 정의서")

# 2. 결과 알림
print(f"✅ 동적 빌드 docx 저장 성공: {path}")
print(f"📄 포함된 요구사항 수: {len(final_reqs)}개")

INFO:services.docx_service:docx 템플릿 정밀 매핑 및 저장 완료: /workspace/agents/req_agent/req_agent/output/사용자 요구사항 정의서_20260604_114837.docx


✅ 동적 빌드 docx 저장 성공: /workspace/agents/req_agent/req_agent/output/사용자 요구사항 정의서_20260604_114837.docx
📄 포함된 요구사항 수: 69개


In [17]:
# pip install docx
!pip install --upgrade python-docx

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [18]:
print(final_reqs[0])
print(repr(final_reqs[0]["description"]))

{'requirement_id': 'COR-001', 'requirement_name': '하도급', 'requirement_type': '제약사항', 'description': '공통 요구사항 \nㅇ 본 사업에서 제시한 사업요건을 준수하여 제안하고, 사업 내용이 변경될 경우 상호 협의 하에 결정 ㅇ 제안요청서 요구사항에 명시되지 않았으나 제안업체의 판단으로 추가가 필요한 요구사항에 대하여 제안서에 별도 표시하여 추가 작성 가능 ㅇ 제안업체는 제안 요청사항을 상세히 검토하여 상기 제안요청 사항의 미비점이나 부족한 사항을 보완하고, 전체 시스템의 완전한 가용성, 안정성, 운영 용이성을 고려하여 제안서를 작성하여야 함 ㅇ 제안요청서의 각 부문에서 제시된 내용의 항목별 수용여부를 제시하여야 하며, 수용하지 못하거나 일부만 수용 가능한 항목에 대하여 사유를 상세히 표기하여야 함 ㅇ 시스템 구축을 위해 필요한 환경설정, 구성변경 등 모든 제반사항은 제안사의 책임으로 수행하되 반드시 진흥원과 협의하여야 함 ㅇ 시스템 이관에 필요한 사항 일체에 대하여 제안업체의 책임으로 수행하여야 함 ㅇ 시스템 설치·연동·이관에 필요한 제조사 및 유지보수 업체의 지원은 제안사가 수행 ㅇ 사업 수행과정에서 발생되는 제반 비용은 제안업체가 부담하여야 하며, 사업 목적 달성을 위해 별도 명시하지 않은 사항이라도 필요한 모든 구성을 제안하여야 함 - 이를 간과하여 사업 수행에 차질이 발생한 경우 별도비용 청구 없이 제안업체의 부담으로 보완 ㅇ 업무 시스템(채널계 포함)과 연동되는 AI 기능은 AI 서비스 장애 또는 중단 시에도 기존 업무 시스템의 정상적인 이용에 영향을 주지 않아야 함 ㅇ AI 기능은 업무 시스템과 비동기 또는 분리 구조로 연계되어야 하며, AI 서비스 미가용 시에도 기존 시스템 기능은 기본 프로세스로 대체 수행될 수 있어야 함', 'source': ['서민금융진흥원 AI기반 통합 플랫폼 구축 사업 제안요청서.docx', 'COR-001', 'ECR-001', 'ECR-002']

In [19]:
for r in rfp:
    if r.get("requirement_id") == "COR-002":
        print(r.get("requirement_name"))
        print(r.get("text") or r.get("description"))

일반사항
일반사항 
ㅇ 기존시스템 운영에 영향을 주지 않아야 하며 기존시스템과 완벽하게 호환되어 정상 동작되어야 함(SSO 연동 포함) ㅇ 사업 중 규정 변경에 따른 연동 대상 시스템 프로세스 변경을 반영 ㅇ 사업 중 발주자의 계획에 따라 생성형 AI를 활용하는 서비스를 개발 시 이와 연동되도록 반영 ㅇ 사업 중 다음을 포함한 의사결정 필요사항은 사전 발주처와 협의해야 함 - 업무시스템에 대한 제반 기술 및 도구 선정 - 화면처리 흐름, 코드분류 등 향후 운영과 유지관리에 영향을 주는 사항 ㅇ 전자정부법 및 개인정보보호법에 따라, 감리 수검 및 개인정보영향평가 대응에 적극적으로 협조하여야 하며, 그 결과에 따라 조치하여야 함 ◦ 본 사업은 제안요청서에 명시되지 아니한 사항이라도 사업목적 달성을 위하여 필요한 사항은 계약상대자와 합의하여 결정할 수 있음 ◦ 발주기관의 방침 또는 사정에 따라 사업수행 내용의 변경이 있을 경우 설계를 변경할 수 있음
